# Exploratory Data Analysis - Previous Application Dataset

EDA on `previous_application.csv` from the Home Credit Default Risk competition: prior loan applications a client made to Home Credit (one row per `SK_ID_PREV`).

**Related Notebooks:**
- `application_eda.ipynb` - Application_train dataset
- `bureau_eda.ipynb` - Bureau dataset

**Analysis Sections:**
1. Setup and Data Loading
2. Basic Dataset Exploration
3. Missing Data Analysis
4. Numeric Features Analysis
5. Categorical Features Analysis
6. Standard Data Quality Checks
7. Functional / Business-Logic Quality Checks

## 1. Setup and Data Loading

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

# Add src to path to import custom modules
sys.path.insert(0, "/Users/bharathregula/code/home_credit_risk_deeplearn/src")
from home_credit_risk.eda_utils import (
    analyze_categorical_features,
    analyze_numeric_features,
    check_categorical_placeholders,
    check_data_quality,
    check_duplicates,
    detect_sentinel_values,
    display_basic_info,
    show_missing_patterns,
)

# Configure polars display options
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(-1)  # show all columns
pl.Config.set_fmt_str_lengths(80)

# Set style for plots
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
# Load dataset
datasets = Path("/Users/bharathregula/code/datasets/home-credit-default-risk")

# Every check below reuses the whole frame, so we collect it once. A large
# infer_schema_length guards against dtype mis-inference across the 37 columns.
prev_df = pl.read_csv(datasets / "previous_application.csv", infer_schema_length=10000)

print("Dataset loaded successfully!")
print(f"Shape: {prev_df.shape}")

In [ ]:
# Quick preview of the data
prev_df.head()

## 2. Basic Dataset Exploration

In [ ]:
# Display basic dataset information
display_basic_info(prev_df, df_name="Previous_Application")

In [ ]:
# Summary statistics
prev_df.describe()

## 3. Missing Data Analysis

Note: `SK_ID_PREV` is the primary key here (one row per previous application), so duplicates are checked against it rather than `SK_ID_CURR` (a client can legitimately have many previous applications).

In [ ]:
# Analyze missing data patterns
missing_summary = show_missing_patterns(prev_df, threshold=0.0)

In [ ]:
# Check for duplicate records (SK_ID_PREV is the primary key)
check_duplicates(prev_df, id_col="SK_ID_PREV")

## 4. Numeric Features Analysis

In [ ]:
# Analyze numeric features
numeric_df = analyze_numeric_features(prev_df, figsize=(18, 6), show_summary=True)

## 5. Categorical Features Analysis

In [ ]:
# Analyze categorical features
analyze_categorical_features(prev_df)

## 6. Standard Data Quality Checks

In [ ]:
# Generic data quality checks
check_data_quality(prev_df)

## 7. Functional / Business-Logic Quality Checks

Cross-field consistency rules that encode how the lending process actually works — beyond per-column sanity. Each check reports a count, percentage and a small sample of offending rows.

In [ ]:
# Small helper for consistent count + percentage reporting
n = prev_df.height


def pct(count: int) -> str:
    return f"{count:,} ({count / n * 100:.3f}%)"


print(f"Total previous applications: {n:,}")

### A. Status-driven cross-field consistency

`NAME_CONTRACT_STATUS` ∈ {Approved, Canceled, Refused, Unused offer}. An *approved* loan is a disbursed contract, so it must carry the terms of that contract (monthly payment, term, credit amount).

In [ ]:
# A1. Approved loans must have a monthly payment (AMT_ANNUITY) and term (CNT_PAYMENT)
approved = prev_df.filter(pl.col("NAME_CONTRACT_STATUS") == "Approved")
a1 = approved.filter(pl.col("AMT_ANNUITY").is_null() | pl.col("CNT_PAYMENT").is_null())
print(
    f"A1. Approved rows missing AMT_ANNUITY or CNT_PAYMENT: {a1.height} "
    f"of {approved.height} approved"
)
a1.select("SK_ID_PREV", "SK_ID_CURR", "AMT_ANNUITY", "CNT_PAYMENT", "AMT_CREDIT").head(
    10
)

In [ ]:
# A2. Approved loans must have a real credit amount (> 0)
a2 = approved.filter(pl.col("AMT_CREDIT").is_null() | (pl.col("AMT_CREDIT") == 0))
print(f"A2. Approved with AMT_CREDIT null or 0: {pct(a2.height)}")
a2.select("SK_ID_PREV", "SK_ID_CURR", "AMT_APPLICATION", "AMT_CREDIT").head(10)

In [ ]:
# A3. Reject-reason consistency.
# XAP means "not rejected": Approved/Canceled use it; Refused must carry a real
# reason. (Unused offer legitimately uses CLIENT, so it is excluded here.)
a3 = prev_df.filter(
    (
        pl.col("NAME_CONTRACT_STATUS").is_in(["Approved", "Canceled"])
        & (pl.col("CODE_REJECT_REASON") != "XAP")
    )
    | (
        (pl.col("NAME_CONTRACT_STATUS") == "Refused")
        & (pl.col("CODE_REJECT_REASON") == "XAP")
    )
)
print(f"A3. Status vs CODE_REJECT_REASON mismatches: {pct(a3.height)}")
a3.select("SK_ID_PREV", "NAME_CONTRACT_STATUS", "CODE_REJECT_REASON").head(10)

### B. Amount relationships

`AMT_APPLICATION` is what the client asked for; `AMT_CREDIT` is what was finally granted. There is **no fixed direction** between them (the approval process can raise or lower the amount), so instead of asserting `applied ≥ credit` we report the split and isolate the `AMT_APPLICATION == 0` population.

In [ ]:
# B4. AMT_APPLICATION vs AMT_CREDIT split
gt = prev_df.filter(pl.col("AMT_APPLICATION") > pl.col("AMT_CREDIT")).height
lt = prev_df.filter(pl.col("AMT_APPLICATION") < pl.col("AMT_CREDIT")).height
eq = prev_df.filter(pl.col("AMT_APPLICATION") == pl.col("AMT_CREDIT")).height
zero = prev_df.filter(pl.col("AMT_APPLICATION") == 0).height
print(f"applied  >  credit : {pct(gt)}")
print(f"applied  <  credit : {pct(lt)}")
print(f"applied  == credit : {pct(eq)}")
print(f"AMT_APPLICATION == 0 (distinct sub-population): {pct(zero)}")

In [ ]:
# Which contract types drive AMT_APPLICATION == 0? (expect cash loans)
prev_df.filter(pl.col("AMT_APPLICATION") == 0).group_by(
    "NAME_CONTRACT_TYPE"
).len().sort("len", descending=True)

In [ ]:
# B5. Down-payment sanity: 0 <= AMT_DOWN_PAYMENT <= AMT_APPLICATION,
#     RATE_DOWN_PAYMENT in [0, 1]
b5 = prev_df.filter(
    (pl.col("AMT_DOWN_PAYMENT") < 0)
    | (pl.col("AMT_DOWN_PAYMENT") > pl.col("AMT_APPLICATION"))
    | (pl.col("RATE_DOWN_PAYMENT") < 0)
    | (pl.col("RATE_DOWN_PAYMENT") > 1)
)
print(f"B5. Down-payment / rate out of bounds: {pct(b5.height)}")
b5.select(
    "SK_ID_PREV", "AMT_APPLICATION", "AMT_DOWN_PAYMENT", "RATE_DOWN_PAYMENT"
).head(10)

### C. Sentinel / placeholder detection (highest value)

Home Credit encodes *not applicable / never* as the magic number `365243` in the `DAYS_*` columns and as `XNA`/`XAP` tokens in categoricals. Neither is stored as null, so they silently corrupt means / describe / correlations.

In [ ]:
# C6. 365243 sentinel across the DAYS_* columns
days_cols = [c for c in prev_df.columns if c.startswith("DAYS_")]
sentinel_summary = detect_sentinel_values(prev_df, columns=days_cols)

In [ ]:
# C7. XNA / XAP placeholder tokens in categorical columns
placeholder_summary = check_categorical_placeholders(prev_df)

### D. Temporal ordering

Milestones of a loan must be chronologically ordered: `DAYS_DECISION ≤ DAYS_FIRST_DUE ≤ DAYS_LAST_DUE ≤ DAYS_TERMINATION`. The `365243` sentinel is nulled out first so it does not create false ordering violations.

In [ ]:
# D8. Temporal ordering (sentinel -> null first)
SENTINEL = 365243
order_cols = ["DAYS_DECISION", "DAYS_FIRST_DUE", "DAYS_LAST_DUE", "DAYS_TERMINATION"]
ordered = prev_df.with_columns(
    [
        pl.when(pl.col(c) == SENTINEL).then(None).otherwise(pl.col(c)).alias(c)
        for c in order_cols
    ]
)

for earlier, later in zip(order_cols, order_cols[1:]):
    v = ordered.filter(pl.col(earlier) > pl.col(later)).height
    print(f"D8. {earlier} > {later}: {pct(v)}")

### E. Flag / duplicate integrity

`FLAG_LAST_APPL_PER_CONTRACT = N` and `NFLAG_LAST_APPL_IN_DAY = 0` mark non-final / repeated applications that may need de-duplication before aggregating to the client level.

In [ ]:
# E9. Repeated-application flags
e9a = prev_df.filter(pl.col("FLAG_LAST_APPL_PER_CONTRACT") == "N").height
e9b = prev_df.filter(pl.col("NFLAG_LAST_APPL_IN_DAY") == 0).height
print(f"E9a. FLAG_LAST_APPL_PER_CONTRACT == N (not last app per contract): {pct(e9a)}")
print(f"E9b. NFLAG_LAST_APPL_IN_DAY == 0 (not last app in day): {pct(e9b)}")